In [1]:
from pathlib import Path 
import urllib.request

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'

if not Path('car_fuel_efficiency_2026.csv').exists():
    urllib.request.urlretrieve(url, 'car_fuel_efficiency_2026.csv')



In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv('car_fuel_efficiency_2026.csv')

df

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0
...,...,...,...,...,...,...,...,...,...,...,...
9995,2005,USA,Gasoline,All-wheel drive,4,2470,6,271.0,4690,18.9,27.4
9996,1993,Asia,Gasoline,All-wheel drive,2,2300,6,244.0,4370,19.0,30.0
9997,2014,USA,Gasoline,All-wheel drive,3,2480,6,267.0,4590,18.4,28.1
9998,2004,USA,Hybrid,Rear-wheel drive,5,2180,6,270.0,4450,20.9,32.2


In [3]:
df.columns

Index(['model_year', 'origin', 'fuel_type', 'drivetrain', 'num_doors',
       'engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight',
       'acceleration', 'fuel_efficiency_mpg'],
      dtype='str')

In [4]:
base_features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']



In [5]:
df['fuel_efficiency_mpg'].min()

np.float64(19.8)

In [6]:
df.isna().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [7]:
df['horsepower'].median()

np.float64(254.0)

In [8]:
def dataset(df, seed: int):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]

    assert len(df_train) + len(df_val) + len(df_test) == len(df)

    return df_train, df_val, df_test


df_train, df_valid, df_test = dataset(df, seed=42)

In [9]:
df_1 = df.fillna(0)
df_1.isna().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [10]:
average = df_train.horsepower.mean()
average

np.float64(254.46338337605272)

In [11]:
df_2 = df.fillna(average)
df_2.isna().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [12]:
def rmse(y_true, y_pred):
    error = np.sqrt(np.mean((y_true - y_pred)**2))
    return error


def linear_regression_model(X_train, y_train, X_valid, y_valid):
    X_train = np.column_stack([np.ones(X_train.shape[0]), X_train])
    w = np.linalg.inv(X_train.T.dot(X_train)).dot(X_train.T).dot(y_train)

    X_valid = np.column_stack([np.ones(X_valid.shape[0]), X_valid])
    y_pred = X_valid @ w
    error = rmse(y_valid, y_pred)
    return w, error, y_pred


df_train_1, df_valid_1, df_test_1 = dataset(df_1, seed=42)
df_train_2, df_valid_2, df_test_2 = dataset(df_2, seed=42)



X_train = df_train_1[base_features].values
y_train = df_train_1['fuel_efficiency_mpg'].values
X_valid = df_valid_1[base_features].values
y_valid = df_valid_1['fuel_efficiency_mpg'].values
w, error, y_pred = linear_regression_model(X_train, y_train, X_valid, y_valid)
w, error, y_pred
print(np.round(error, 3))
print(y_valid[:5], y_pred[:5])

2.205
[30.4 32.8 27.8 28.6 31.1] [31.64502134 29.86572978 31.58838899 30.02584068 31.006497  ]


In [13]:
df_train_1, df_valid_1, df_test_1 = dataset(df_1, seed=42)
df_train_2, df_valid_2, df_test_2 = dataset(df_2, seed=42)



X_train = df_train_2[base_features].values
y_train = df_train_2['fuel_efficiency_mpg'].values
X_valid = df_valid_2[base_features].values
y_valid = df_valid_2['fuel_efficiency_mpg'].values
w, error, y_pred = linear_regression_model(X_train, y_train, X_valid, y_valid)
w, error, y_pred
print(np.round(error, 3))
print(y_valid[:5], y_pred[:5])

2.202
[30.4 32.8 27.8 28.6 31.1] [31.54288381 29.92505122 31.56459523 29.95461814 31.00171448]


In [14]:
def regularized_linear_regression(X_train, y_train, X_valid, y_valid, r):
    X_train = np.column_stack([np.ones(X_train.shape[0]), X_train])

    R = np.eye(X_train.shape[1]) * r
    R[0, 0] = 0
    w = np.linalg.inv(X_train.T.dot(X_train) + R).dot(X_train.T).dot(y_train)

    X_valid = np.column_stack([np.ones(X_valid.shape[0]), X_valid])
    y_pred = X_valid @ w
    error = rmse(y_valid, y_pred)
    return w, error, y_pred

In [19]:
X_train = df_train_1[base_features].values
y_train = df_train_1['fuel_efficiency_mpg'].values
X_valid = df_valid_1[base_features].values
y_valid = df_valid_1['fuel_efficiency_mpg'].values

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    _, error, _ = regularized_linear_regression(X_train, y_train, X_valid, y_valid, r)
    print(r, ': ', error)


0 :  2.205293194751098
0.01 :  2.2052931948017465
0.1 :  2.2052931952047836
1 :  2.20529319930982
5 :  2.2052932175553184
10 :  2.2052932403806085
100 :  2.205293654148663


In [20]:
X_train = df_train_2[base_features].values
y_train = df_train_2['fuel_efficiency_mpg'].values
X_valid = df_valid_2[base_features].values
y_valid = df_valid_2['fuel_efficiency_mpg'].values

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    _, error, _ = regularized_linear_regression(X_train, y_train, X_valid, y_valid, r)
    print(r, ': ', error)

0 :  2.2018174845681155
0.01 :  2.2018174844357423
0.1 :  2.201817483227154
1 :  2.2018174711310867
5 :  2.2018174174077205
10 :  2.2018173502602973
100 :  2.201816144433746


In [28]:
df_train, df_valid, df_test = dataset(df_1, seed=42)

# print(df_1.isna().sum())

r = 0.001

df_train_full = pd.concat([df_train, df_valid])

print(df_train_full.isna().sum())

print(len(df_train_full))

X_train = df_train_full[base_features].values
y_train = df_train_full['fuel_efficiency_mpg'].values
X_test = df_test[base_features].values
y_test = df_test['fuel_efficiency_mpg'].values


w, error, pred = regularized_linear_regression(X_train, y_train, X_test, y_test, r)
print(error)

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64
8000
2.2382656729954498
